![Banner](img/AI_Special_Program_Banner.jpg)

# Introduction to LLMs - Exercise
---
Instructions are given in $\color{teal}{\textsf{teal}}$ color.

<u><b style='color:green'>Hint:</b> It might make sense to split your solution into separate notebooks, in particular tasks 1, 3, and 4 should probably each have their **own notebook**, which you can link here.</u>

---

## Solution overview

Following the hint above, tasks 1 and 3 each live in their own notebook, and task 4 is solved directly
in the material notebook it refers to. Task 2 is answered below.

| Task | Solution | Runs against |
|---|---|---|
| 1 - RAG to answer questions | [3.4_LC_Ex_Task1_RAG.ipynb](3.4_LC_Ex_Task1_RAG.ipynb) | `gemma3` + `embeddinggemma` |
| 2 - Components of a RAG system | [below](#Task-2) | - (written) |
| 3 - Compare two open source LLMs | [3.4_LC_Ex_Task3_Compare.ipynb](3.4_LC_Ex_Task3_Compare.ipynb) | `qwen2.5:1.5b`, `deepseek-r1:1.5b`, `qwen2.5:0.5b` |
| 4 - LLM for sentiment analysis | [3.4_6_LC_Ollama_Sentiment_HandsOn.ipynb](3.4_6_LC_Ollama_Sentiment_HandsOn.ipynb) | `gemma2:2b`, `qwen2.5:1.5b`, `gemma3` |

All models are served by a local Ollama instance at `http://localhost:11434`. They were chosen to fit
on a laptop CPU, which is why the "big" model in task 4 is `gemma3` (4B) rather than the `gemma4:12b`
suggested there - see the note in that notebook.

**Prerequisites** for re-running everything:

```bash
ollama pull gemma3            # the default tag IS the 4B variant
ollama pull embeddinggemma    # the embedding model for the RAG task
ollama pull gemma2:2b
ollama pull qwen2.5:1.5b
ollama pull qwen2.5:0.5b
ollama pull deepseek-r1:1.5b
```

```bash
pip install langchain langchain-community langchain-ollama langchain-text-splitters ollama faiss-cpu "arxiv<3" pymupdf
```

The pin on `arxiv` is deliberate: with `arxiv` 4.x, `langchain-community`'s `ArxivLoader` calls
`Search.results()`, which was removed from the package, and fails with an `AttributeError`. Task 1
therefore loads the paper with `PyPDFLoader` and only uses `ArxivLoader` for comparison.

### $\color{teal}{\textsf{Task 1: Use RAG to answer questions}}$

Use your local Ollama LLM server to answer the following questions:

* What does self-attention do in the context of transformers?
* What are positional encodings and why are they needed?

**Hints**: 
* The answers may be found in the original [transformer paper](https://arxiv.org/abs/1706.03762) and
* you must find a suitable document loader (see the link in the [RAG notebook](3.4_4_LC_RAG_Ollama.ipynb)).

$\color{teal}{\textsf{Solution:}}$ **[3.4_LC_Ex_Task1_RAG.ipynb](3.4_LC_Ex_Task1_RAG.ipynb)**

The suitable document loader is the point of the task: the source is a **PDF** on arXiv, so
`WebBaseLoader` (which strips tags out of an HTML page) is the wrong tool. `ArxivLoader` takes the
arXiv ID directly, `PyPDFLoader` is the generic alternative and also accepts a URL. Both are used
there, the second one as a fallback.

### $\color{teal}{\textsf{Task 2: What components is a RAG system made up of?}}$

* As a starting point, see the image in the [RAG notebook](3.4_4_LC_RAG_Ollama.ipynb)
* Start by thinking about the use case where a company wants to make all its service manuals available to its service technicians by way of a chatbot in order to decide what the main difference is between the notebook and a real RAG system
* When describing the components, separate them into AI and non-AI components

$\color{teal}{\textsf{Components of a RAG system:}}$

<a id="Task-2"></a>

The picture in the [RAG notebook](3.4_4_LC_RAG_Ollama.ipynb) shows the *query* path only - and the
notebook itself points that out: the **data preparation phase is missing** from it. That observation is
already most of the answer, because it is exactly where the notebook and a real system diverge.

### The main difference between the notebook and a real RAG system

In the notebook, indexing and querying are the *same* program: `WebBaseLoader` fetches one document,
`FAISS.from_documents` embeds it, and three cells later we ask a question. One document, one language,
one user, one run, everything in memory.

A production system splits this into **two systems with completely different requirements**:

* an **offline ingestion pipeline** (batch, throughput-oriented, runs on a schedule): find documents,
  parse them, chunk them, embed them, write them into an index - and, crucially, do that *again*
  whenever a document changes, appears or is withdrawn;
* an **online query service** (request/response, latency- and availability-oriented): retrieve, assemble
  the prompt, generate, cite, and log.

So the main difference is not the LLM - the LLM is the one part the notebook already gets right. It is
the **document lifecycle, the retrieval quality machinery and everything around the models** that is
missing. Put differently: the notebook demonstrates the *idea* of RAG; a real system is 10% idea and
90% data engineering.

### The service manual use case

Making all service manuals available to technicians through a chatbot sharpens this considerably:

* **The documents are hard, not the questions.** Manuals are PDFs full of tables, exploded drawings,
  torque specifications and part numbers - and a naive text extractor turns a table into an unreadable
  sequence of numbers. Parsing quality, not model quality, sets the ceiling of the whole system.
* **Version and machine type are safety-critical.** The same procedure differs between machine
  revisions. Retrieving the right paragraph from the *wrong* manual produces a fluent, confident and
  dangerous answer. Every chunk therefore needs metadata (machine type, serial range, manual version,
  revision date, language), and retrieval has to *filter* on it before it ranks anything.
* **A wrong answer has physical consequences.** The system must cite manual and page so the technician
  can verify, and "I could not find this in the manuals" must be an acceptable answer rather than
  something the model talks its way out of.
* **Not every technician may see every document.** Access rights have to be applied *during* retrieval,
  not by filtering the answer afterwards - by then the content has already been in the prompt.
* **The corpus lives.** New machines, new firmware releases, withdrawn revisions - the index has to
  follow, incrementally, and a withdrawn manual must actually disappear.
* **The setting is a workshop**, so: multilingual, mobile, possibly offline, and the answer has to be
  fast enough to be used while standing at the machine.

### The components

$\color{teal}{\textsf{AI components (a model does the work):}}$

| Component | Role | In the notebook? |
|---|---|---|
| **Embedding model** | turns chunks *and* queries into vectors; the same model must be used for both, and changing it invalidates the whole index | yes (`embeddinggemma`) |
| **Generator LLM** | writes the answer from the retrieved context | yes (`gemma3`) |
| **Re-ranker** (cross-encoder) | re-scores the top ~50 candidates against the query; the single most effective quality lever after chunking | no |
| **Query transformation** | an LLM rewrites the technician's question (resolves "it", expands abbreviations, splits multi-part questions, translates) before retrieval | no |
| **Document understanding / OCR / layout models** | make scanned manuals, tables and diagrams machine-readable; a multimodal model if drawings are to be answered about | no |
| **Guardrail & evaluation models** | groundedness check ("is this answer supported by the retrieved chunks?"), PII / prompt-injection detection, LLM-as-a-judge in the regression suite | no |
| **Router / intent classifier** | decides whether a question needs retrieval at all, or which corpus to search | no |

$\color{teal}{\textsf{Non-AI components (ordinary software and data engineering):}}$

| Component | Role | In the notebook? |
|---|---|---|
| **Source connectors** | file shares, DMS/PLM, SharePoint, ticket system - plus change detection, so only what changed is re-processed | one `WebBaseLoader` call |
| **Document parsers / loaders** | PDF, DOCX, HTML, XML to text, *including* tables and reading order | rudimentary |
| **Chunking / text splitter** | deterministic code; size and overlap decide what can be retrieved at all | yes (`RecursiveCharacterTextSplitter`) |
| **Metadata extraction & schema** | machine type, version, language, confidentiality, page number - the basis for filtering and for citations | no |
| **Vector database** | persistent, updatable, filterable index (Qdrant, pgvector, Elasticsearch) | FAISS in memory |
| **Lexical index (BM25) + hybrid search** | part numbers and error codes are *strings*; vector search is bad at exact matches, keyword search is good at them | no |
| **Retrieval logic** | metadata filters, top-k, MMR, deduplication, context budget | partly (`as_retriever`) |
| **Prompt assembly / orchestration** | templates, context window budgeting, conversation history, tool calls | yes (LCEL chain) |
| **AuthN / AuthZ** | who may see which document - enforced *inside* the retrieval query | no |
| **Application layer** | chat UI, sessions, feedback buttons, mobile/offline client | no |
| **Citation & source display** | link back to manual, revision and page | no |
| **Serving infrastructure** | model server, GPU capacity, batching, caching, autoscaling | Ollama on one machine |
| **Evaluation harness** | a golden set of question/answer/source triples, run on every change to chunking, embedding or prompt | no |
| **Monitoring, logging, cost tracking** | latency, retrieval hit rate, "not found" rate, user feedback | no |
| **Governance & versioning** | index version tied to embedding-model version, reproducible rebuilds, rollback, retention | index directory name only |

### What this says

Counting the rows is the point: **two** of the components are indispensably AI, and one of those two
(the embedding model) is not even the LLM. Everything else is search engineering, data engineering and
ordinary application development. The reason RAG projects fail is almost never the language model -
it is unparsed tables, missing metadata, a stale index or the absence of any way to tell whether the
retrieval got better or worse.

The notebook is honest about this in its own way: its most instructive moment is not the answer the
model produces, but the observation that a rephrased question retrieves different chunks and therefore
yields a different answer. In production, that sentence turns into the evaluation harness.

### $\color{teal}{\textsf{Task 3: Compare two open source LLMs}}$

Re-do the HandsOn Exercise from [the first notebook](3.4_1_LC_local_HandsOn.ipynb) using your Ollama installation with your versions of Qwen2.5 and Deepseek-R1. Concretely, use the local LLMs to find out what the difference between open weights and open source LLMs is! What do you observe with respect to the responses of the two models? What are the reasons in your opinion?

$\color{teal}{\textsf{Solution:}}$ **[3.4_LC_Ex_Task3_Compare.ipynb](3.4_LC_Ex_Task3_Compare.ipynb)**

All three parts of the original HandsOn Exercise are re-done there against Ollama:
open weights vs. open source, the Luke Littler question, and the comparison with the `0.5B` sibling.

### $\color{teal}{\textsf{Task 4: Use an LLM for Sentiment Analysis}}$
If you have not yet done so, finish the HandsOn Exercise from the [last one of the material notebooks](3.4_6_LC_Ollama_Sentiment_HandsOn.ipynb).

$\color{teal}{\textsf{Solution:}}$ **[3.4_6_LC_Ollama_Sentiment_HandsOn.ipynb](3.4_6_LC_Ollama_Sentiment_HandsOn.ipynb)**

The HandsOn Exercise is solved in that notebook: three models classify the same 50 reviews, the results
are collected in a dataframe against the ground truth, the metrics come from the confusion matrix, the
time per review is measured and extrapolated to all 50,000 - and the whole thing is compared against a
TF-IDF + logistic regression baseline, which is what actually answers the question whether LLMs are the
way to go here.